## Notebook Description

The notebook will contain 5 main components:
- Configuration
- Data collection & cleaning
- Feature creation
- Model training
- Results

Each main component may contain mutiple subcomponents depending on the complexity of the component
The novelty added in this notebook is the usage of three new features:
- Quali Position
- Historical influence of quali result to the final race position on each track
- FP2 pace

Disclaimer: This is a notebook that uses as a starting point the Austin_GP_Winner_Predictor made by Mariana Antaya


### Step 0 - Package Installation

In [ ]:
# %pip install fastf1 scikit-learn xgboost matplotlib

### Step 1 - Configuration

- Import the necessary libraries 
- Configure different necessities as cache

In [16]:
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import time
import fastf1
from fastf1.ergast import Ergast
import sklearn
#from sklearn.preprocessing import PolynomialFeatures
import matplotlib.pyplot as plt

cache_dir = Path('data_cache')
cache_dir.mkdir(exist_ok=True)
fastf1.Cache.enable_cache(str(cache_dir))

api = Ergast(result_type='pandas')

YEAR = 2025
START_YEAR = 2022

### Step 2 - Data Collection

This step will be split into two substeps:
- Step 2.1 Data Collection & Cleaning as in reference notebook
- Step 2.2 Data Collection & Cleaning for new features
- Step 2.3 Final Data Aggregation

#### Step 2.1 Data Collection & Cleaning - ref notebook

In [25]:
def season_results(year):
    csv_path = cache_dir / f"results_{year}.csv"

    page = api.get_race_results(season=year)

    features = ['season', 'round', 'raceDate', 'circuitId', 'raceName']
    chunks = []
    received = 0

    while True:
        for (_, meta), result in zip(page.description.iterrows(), page.content):
            frame = pd.DataFrame(result).copy()
            for feature in features:
                frame[feature] = meta[feature]
            chunks.append(frame)
            
        try:
            time.sleep(0.4)
            page = page.get_next_result_page()
        except ValueError as exc:
            if 'No more data' in str(exc):
                break
            raise

    if not chunks:
        return pd.DataFrame()

    frame = pd.concat(chunks, ignore_index=True).drop_duplicates(['season', 'round', 'driverId'])
    frame.to_csv(csv_path, index=False)
    return frame

schedule = api.get_race_schedule(season=YEAR)
austin = schedule.loc[schedule['circuitId'].eq('americas')]
if len(austin) != 1:
    raise ValueError("No unique Austin race in this season schedule. Check target year")

target = austin.iloc[0]
target_date = pd.Timestamp(target['raceDate']).tz_localize(None).normalize()

frames = []
for year in range(START_YEAR, YEAR):
    part = season_results(year)
    if not part.empty:
        frames.append(part)
    print(f"{year}: {len(part)} driver results loaded")

raw = pd.concat(frames, ignore_index=True)
raw['raceDate'] = pd.to_datetime(raw['raceDate']).dt.tz_localize(None)
today = pd.Timestamp(datetime.now(timezone.utc).date())

history = raw.loc[(raw['raceDate'] < target_date) & (raw['raceDate'] < today)].copy()
history['position'] = pd.to_numeric(history['position'], errors='raise')
history['points'] = pd.to_numeric(history['points'], errors='raise')
history = history.sort_values(['raceDate', 'season', 'round', 'position'])

history.head()



2022: 440 driver results loaded
2023: 440 driver results loaded
2024: 479 driver results loaded


,number,position,positionText,points,grid,laps,status,driverId,driverNumber,driverCode,...,fastestLapRank,fastestLapNumber,fastestLapTime,fastestLapAvgSpeedUnits,fastestLapAvgSpeed,season,round,raceDate,circuitId,raceName
0,16,1,1,26.0,1,57,Finished,leclerc,16,LEC,...,1.0,51.0,0 days 00:01:34.570000,kph,206.018,2022,1,2022-03-20,bahrain,Bahrain Grand Prix
1,55,2,2,18.0,3,57,Finished,sainz,55,SAI,...,3.0,52.0,0 days 00:01:35.740000,kph,203.501,2022,1,2022-03-20,bahrain,Bahrain Grand Prix
2,44,3,3,15.0,5,57,Finished,hamilton,44,HAM,...,5.0,53.0,0 days 00:01:36.228000,kph,202.469,2022,1,2022-03-20,bahrain,Bahrain Grand Prix
3,63,4,4,12.0,9,57,Finished,russell,63,RUS,...,6.0,56.0,0 days 00:01:36.302000,kph,202.313,2022,1,2022-03-20,bahrain,Bahrain Grand Prix
4,20,5,5,10.0,7,57,Finished,kevin_magnussen,20,MAG,...,8.0,53.0,0 days 00:01:36.623000,kph,201.641,2022,1,2022-03-20,bahrain,Bahrain Grand Prix


#### Step 2.2 Data Collection & Cleaning - ref notebook

### Step 2.3 Final Data Aggregation

### Step 3 Features Extraction and Creation

### Step 4 Model Creation

### Step 5 Results